# 02 - Preparar paths destino para datastore

Lee la carpeta de imagenes normalizadas, calcula fecha, sector espacial, nombre oficial y ruta destino en el datastore.

In [ ]:
from pathlib import Path
import importlib
import json
import sys

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'core').exists() and (candidate / 'flujo_geosupport_final' / 'settings.json').exists():
            return candidate
        if candidate.name.lower() == 'flujo_geosupport_final' and (candidate / 'settings.json').exists():
            parent = candidate.parent
            if (parent / 'core').exists():
                return parent
    raise RuntimeError('No se pudo resolver PROJECT_ROOT: debe existir core/ y flujo_geosupport_final/settings.json')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import flujo_geosupport_final.scripts.geosupport_flujo_completo as flujo
flujo = importlib.reload(flujo)

FINAL_DIR = PROJECT_ROOT / 'flujo_geosupport_final'
SETTINGS_PATH = FINAL_DIR / 'settings.json'
with SETTINGS_PATH.open('r', encoding='utf-8') as file_obj:
    SETTINGS = json.load(file_obj)
flujo.apply_flow_settings(SETTINGS)

def resolve_path(value):
    return flujo.resolve_config_path(value)

## Parametros

In [ ]:
# Carpeta creada por la etapa 01. Debe contener los TIFF normalizados.
INPUT_FOLDER_NORMALIZADAS = resolve_path(SETTINGS['input_folder_tifs']) / SETTINGS.get('normalized_folder_name', 'normalizadas')

# Fechas externas con prioridad. Primero se busca aqui (JSON/Excel); si no existe, se usa regex sobre el nombre.
DATE_JSON = resolve_path(SETTINGS.get('date_json'))
DATE_EXCEL = resolve_path(SETTINGS.get('date_excel'))

OUTPUT_DIR = FINAL_DIR / 'outputs' / '02_preparar_paths_datastore'

print('Settings:', SETTINGS_PATH)
print('Entrada normalizada:', INPUT_FOLDER_NORMALIZADAS)
print('JSON fechas:', DATE_JSON)
print('Excel fechas:', DATE_EXCEL)
print('Datastore root:', flujo.DATASTORE_ROOT)
print('Feature indice:', flujo.PATH_FC_FOOTPRINT_INDICE)
print('Salida:', OUTPUT_DIR)

## Ejecutar preparacion

In [ ]:
stage = flujo.prepare_paths_stage(
    input_folder=INPUT_FOLDER_NORMALIZADAS,
    output_dir=OUTPUT_DIR,
    date_json=DATE_JSON,
    date_excel=DATE_EXCEL,
)

display(stage['summary_df'])
display(stage['ready_df'].head(20))
display(stage['review_df'].head(20))